# 04. Forward & Backward Chaining (Horn Clause)

Notebook 02 mengecek entailment dengan mengenumerasi seluruh model -- cara
ini sound dan complete, tapi untuk KB dengan $n$ simbol jumlah modelnya
$2^n$. Notebook ini mengenalkan cara yang jauh lebih murah kalau KB-nya
berbentuk khusus, yaitu Horn clause: dua algoritma inferensi, forward
chaining dan backward chaining, yang berjalan dalam waktu linear terhadap
ukuran KB, bukan eksponensial.

Setelah menyelesaikan notebook ini, Anda diharapkan mampu:

1. mengenali apakah sebuah sentence merupakan Horn clause / definite
   clause;
2. menjelaskan Modus Ponens sebagai inference rule, dan mengaitkannya ke
   notasi $KB \vdash_i \alpha$ dari Notebook 02;
3. menelusuri (trace) forward chaining langkah demi langkah pada sebuah
   Horn-clause KB; dan
4. menelusuri backward chaining pada KB yang sama, termasuk kenapa perlu
   penjagaan terhadap infinite loop, lalu membandingkannya dengan forward
   chaining.

## Setup

Jalankan sel di bawah ini sekali di awal, sebelum sel mana pun yang lain.

Sel ini memasang dependensi yang diperlukan, mencari folder yang berisi
`logic.py` dan `utils.py`, lalu mengimpornya. Kalau notebook dibuka lewat Google
Colab, repo akan di-clone otomatis. Tidak ada yang perlu diubah di sini.

Environment sudah siap kalau baris terakhir output mencetak
`Check       : tt_entails(P & Q, Q) = True`.

In [1]:
# =============================================================================
# Standard setup cell.
# Run this once, before any other cell in this notebook.
# =============================================================================
import importlib.util
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/kcv-if/Modul-Praktikum-KK-RKA-25.git"
ON_COLAB = "google.colab" in sys.modules


def ensure_dependencies():
    """Install only the packages this module actually uses."""
    required = {
        "networkx": "networkx",
        "numpy": "numpy",
        "pandas": "pandas",
        "matplotlib": "matplotlib",
        "ipywidgets": "ipywidgets",
        "PIL": "pillow",
        "pygments": "pygments",
    }
    missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
    if missing:
        print("Installing:", ", ".join(missing))
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)


def find_environment(start):
    """Locate the folder that holds logic.py and utils.py, searching upward."""
    for root in [start, *start.parents]:
        for candidate in sorted(root.rglob("logic.py")):
            if (candidate.parent / "utils.py").exists():
                return candidate.parent
        if (root / ".git").exists():
            break
    return None


ensure_dependencies()

start_dir = Path.cwd()
if ON_COLAB:
    clone_dir = Path("Modul-Praktikum-KK-RKA-25")
    if not clone_dir.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(clone_dir)], check=True)
    start_dir = clone_dir

ENV_DIR = find_environment(start_dir)
if ENV_DIR is None:
    raise RuntimeError(
        "Environment folder not found. Make sure this notebook is opened from "
        "inside the Modul-Praktikum-KK-RKA-25 repository."
    )
if str(ENV_DIR) not in sys.path:
    sys.path.insert(0, str(ENV_DIR))

import itertools
import warnings

import pandas as pd

# qpsolvers is only used by the SVM code, which this module never touches.
warnings.filterwarnings("ignore", message="no QP solver found")

from logic import *
from notebook import psource
from utils import *

print("Environment :", ENV_DIR)
print("Python      :", sys.version.split()[0])
print("Check       : tt_entails(P & Q, Q) =", tt_entails(expr("P & Q"), expr("Q")))

Environment : C:\Users\cathl\Kuliah\KCV\KK\Modul-Praktikum-KK-RKA-25\logics\praktikum\environment
Python      : 3.14.2
Check       : tt_entails(P & Q, Q) = True


---
# 4.1 Inference Rules & Modus Ponens

## Penjelasan

Sejauh ini entailment ($KB \vDash \alpha$, Notebook 02) selalu dicek dengan
**model checking**: enumerasi seluruh model, lalu periksa apakah $\alpha$
benar di semua model tempat $KB$ benar. Cara ini pasti benar, tapi mahal:
$2^n$ model untuk $n$ simbol.

Alternatifnya adalah **inference rule**: aturan yang menurunkan sentence
baru langsung dari bentuk (syntax) sentence yang sudah ada, tanpa perlu
menyentuh model sama sekali. Proses menurunkan sentence lewat serangkaian
inference rule disebut **proof**, dan notasi $KB \vdash_i \alpha$ dari
Notebook 02 (bagian Sound dan complete) persis menangkap ini: $\alpha$
diturunkan dari $KB$ lewat prosedur $i$.

Inference rule paling dasar, dan yang jadi fondasi seluruh notebook ini,
adalah **Modus Ponens**:

$$\frac{\alpha \Rightarrow \beta, \quad \alpha}{\beta}$$

Dibaca: kalau $\alpha \Rightarrow \beta$ dan $\alpha$ sama-sama ada di
$KB$, maka $\beta$ boleh disimpulkan. Forward chaining (4.3) dan backward
chaining (4.4) pada dasarnya adalah dua cara berbeda untuk mengaplikasikan
Modus Ponens berulang kali sampai query terjawab.

## Contoh penerapan

Modus Ponens sendiri gampang diverifikasi: klaim kesimpulannya harus
selalu cocok dengan `tt_entails` dari Notebook 02/03, karena keduanya
sama-sama menjawab pertanyaan yang sama ($KB \vDash \alpha$), cuma beda
cara.

In [2]:
# Modus Ponens: dari (P ==> Q) dan P, simpulkan Q. Verifikasi klaim ini
# lewat tt_entails (model checking dari Notebook 02/03) -- kalau Modus
# Ponens sound, keduanya harus selalu setuju.
P, Q = expr('P, Q')

kb = P & (P |'==>'| Q)
conclusion = Q

print("KB                :", kb)
print("Modus Ponens ->   :", conclusion)
print("tt_entails setuju?:", tt_entails(kb, conclusion))

KB                : (P & (P ==> Q))
Modus Ponens ->   : Q
tt_entails setuju?: True


---
# 4.2 Horn Clause & Definite Clause

## Penjelasan

Modus Ponens bisa diterapkan berulang secara efisien kalau seluruh KB
dibatasi ke bentuk khusus yang disebut **Horn clause**: sebuah clause
(disjunction of literal) dengan **paling banyak satu** literal positif.
Ditulis dalam bentuk implikasi, ada dua variannya:

- **Fact**: proposition symbol tunggal, misalnya $A$.
- **Rule**: $(\text{Symbol}_1 \land \dots \land \text{Symbol}_n)
  \Rightarrow \text{Symbol}$, dengan seluruh premise dan konklusinya
  proposition symbol positif (bukan negasi).

Bentuk seperti ini, khusus yang punya **tepat satu** literal positif,
disebut **definite clause**. Semua definite clause adalah Horn clause,
tapi tidak sebaliknya (Horn clause juga membolehkan nol literal positif,
dipakai untuk menyatakan "goal" yang harus dibantah, di luar cakupan
notebook ini).

Kenapa dibatasi ke bentuk ini? Karena forward dan backward chaining pada
KB yang seluruhnya definite clause berjalan dalam waktu **linear**
terhadap ukuran KB, jauh lebih murah dibanding model checking yang
eksponensial. Ongkosnya, tidak semua sentence propositional logic bisa
ditulis sebagai definite clause (disjunction seperti $P \lor Q$ tanpa
negasi, misalnya, tidak bisa). Metode yang tidak punya batasan ini ada,
namanya **resolution**, tapi cakupannya di luar notebook ini.

**Reuse contoh dari slide.** Rule set berikut dipakai berulang sepanjang
notebook ini:

$$P \Rightarrow Q,\quad L \land M \Rightarrow P,\quad B \land L
\Rightarrow M,\quad A \land P \Rightarrow L,\quad A \land B \Rightarrow L,
\quad A,\quad B$$

Rule set ini sudah tersedia langsung di `logic.py` sebagai
`horn_clauses_KB` (sebuah `PropDefiniteKB`), jadi tidak perlu ditulis
ulang.

## Contoh penerapan

`is_definite_clause` mengecek apakah sebuah sentence sesuai bentuk di
atas; `parse_definite_clause` memecahnya jadi daftar premise dan satu
konklusi.

In [3]:
psource(is_definite_clause, parse_definite_clause)

In [4]:
# horn_clauses_KB: rule set Figure 7.16, sudah tersedia di logic.py.
for clause in horn_clauses_KB.clauses:
    print(clause, "-> definite clause?", is_definite_clause(clause))

(P ==> Q) -> definite clause? True
((L & M) ==> P) -> definite clause? True
((B & L) ==> M) -> definite clause? True
((A & P) ==> L) -> definite clause? True
((A & B) ==> L) -> definite clause? True
A -> definite clause? True
B -> definite clause? True


In [5]:
# parse_definite_clause memecah satu rule jadi (premises, conclusion).
rule = expr('(A & B) ==> L')
premises, conclusion = parse_definite_clause(rule)
print("Rule      :", rule)
print("Premises  :", premises)
print("Conclusion:", conclusion)

Rule      : ((A & B) ==> L)
Premises  : [A, B]
Conclusion: L


---
# 4.3 Forward Chaining

## Penjelasan

**Forward chaining** bersifat *data-driven*: mulai dari fact yang sudah
diketahui, lalu berulang kali mencari rule yang seluruh premise-nya sudah
terpenuhi, menambahkan konklusinya sebagai fact baru, sampai tidak ada
lagi fact baru yang bisa ditambahkan (atau query sudah ditemukan).

Algoritma PL-FC-ENTAILS (Figure 7.15) melacak tiga hal:

- **agenda**: daftar fact yang belum diproses.
- **count[c]**: berapa banyak premise rule $c$ yang *belum* terpenuhi.
  Begitu `count[c]` mencapai nol, konklusi $c$ boleh ditambahkan.
- **inferred[p]**: apakah fact $p$ sudah pernah diproses, supaya tidak
  diproses ulang.

Sudah diimplementasikan di `logic.py` sebagai `pl_fc_entails(kb, q)`,
tapi fungsi ini cuma mengembalikan `True`/`False`, tanpa menunjukkan
langkah-langkahnya.

## Contoh penerapan

Lihat dulu source code aslinya, lalu jalankan pada `horn_clauses_KB`
untuk membuktikan `Q`.

In [6]:
psource(pl_fc_entails)

In [7]:
print("Q entailed?", pl_fc_entails(horn_clauses_KB, expr('Q')))

Q entailed? True


`pl_fc_entails` membuktikan `Q`, tapi tidak menunjukkan urutan rule yang
menyala (*fired*) untuk sampai ke sana -- padahal justru itu yang paling
penting untuk dipahami. Tulis ulang algoritmanya (source code yang sama
persis, tinggal ditambah pencatatan) supaya tiap langkah tercatat sebagai
baris tabel, cocok dengan trace di slide.

In [8]:
def pl_fc_entails_traced(kb, q):
    """Like pl_fc_entails (Figure 7.15), but also builds a step-by-step
    trace as a DataFrame, mirroring the AGENDA/COUNT/INFERRED walkthrough
    from the slide."""
    count = {c: len(conjuncts(c.args[0])) for c in kb.clauses if c.op == '==>'}
    inferred = {}
    agenda = [s for s in kb.clauses if is_prop_symbol(s.op)]
    rows = []
    while agenda:
        p = agenda.pop()
        if p == q:
            rows.append({"processed": str(p), "rule fired": "-", "new fact": "query found"})
            return True, pd.DataFrame(rows)
        if not inferred.get(p, False):
            inferred[p] = True
            fired = []
            for c in kb.clauses_with_premise(p):
                count[c] -= 1
                if count[c] == 0:
                    agenda.append(c.args[1])
                    fired.append((str(c), str(c.args[1])))
            rows.append({
                "processed": str(p),
                "rule fired": ", ".join(r for r, _ in fired) or "-",
                "new fact": ", ".join(f for _, f in fired) or "-",
            })
    return False, pd.DataFrame(rows)


found, trace = pl_fc_entails_traced(horn_clauses_KB, expr('Q'))
print("Q entailed?", found)
trace

Q entailed? True


,processed,rule fired,new fact
0,B,-,-
1,A,((A & B) ==> L),L
2,L,((B & L) ==> M),M
3,M,((L & M) ==> P),P
4,P,"(P ==> Q), ((A & P) ==> L)","Q, L"
5,Q,-,query found


Baris `processed` menunjukkan fact mana yang sedang diproses; `rule
fired` menunjukkan rule yang langsung menyala begitu fact itu diproses
(count-nya turun ke nol); `new fact` adalah konklusi yang ditambahkan ke
agenda. Urutan persisnya (`A`, `B`, lalu rule mana dulu yang menyala)
tergantung urutan `agenda.pop()` mengambil dari akhir list, tapi
konklusi akhirnya sama: begitu `P` diproses, rule `P ==> Q` langsung
menyala dan `Q` ditemukan.

---
# 4.4 Backward Chaining

## Penjelasan

**Backward chaining** bersifat *goal-driven*: kebalikan arah dari forward
chaining. Mulai dari query (goal), lalu:

1. kalau goal sudah berupa fact yang diketahui, langsung `True`;
2. kalau tidak, cari rule yang konklusinya cocok dengan goal, lalu coba
   buktikan **setiap** premise rule itu secara rekursif sebagai sub-goal
   baru;
3. kalau semua sub-goal terbukti, goal awal ikut terbukti.

Karena sifatnya rekursif dan rule bisa saling merujuk, backward chaining
butuh penjagaan terhadap **infinite loop**: kalau sedang mencoba
membuktikan goal $g$, dan di tengah jalan $g$ muncul lagi sebagai sub-goal
dari dirinya sendiri (misalnya $A \Rightarrow B, B \Rightarrow C, C
\Rightarrow A$, lalu ditanya buktikan $A$), rekursinya tidak akan pernah
berhenti kalau tidak ada yang mencatat goal mana saja yang sedang dalam
proses pembuktian di jalur yang sama. `logic.py` tidak menyediakan versi
propositional dari backward chaining, jadi ditulis sendiri di bawah,
termasuk penjagaan loop-nya.

Forward dan backward chaining sama-sama mengaplikasikan Modus Ponens
berulang kali, tapi dari arah berlawanan:

| | Forward chaining | Backward chaining |
|---|---|---|
| Titik mulai | Fact | Query (goal) |
| Arah | Fact $\to$ goal | Goal $\to$ fact |
| Sifat | Data-driven | Goal-driven |
| Perlu penjagaan loop? | Tidak (tiap fact cuma diproses sekali lewat `inferred`) | Ya (goal bisa muncul lagi di jalur rekursi yang sama) |

## Contoh penerapan

Implementasi backward chaining di bawah mengikuti pseudocode
BACKWARD_CHAIN: cek fact, cari rule dengan konklusi yang cocok, rekursi
ke tiap premise, dengan `visited` sebagai penjagaan loop.

In [9]:
def pl_bc_ask(kb, goal, visited=None, trace=None):
    """Backward chaining for a PropDefiniteKB. Returns True/False.
    `visited` guards against infinite loops on cyclic rule sets: it tracks
    goals currently being proved along *this* recursive path, not
    globally. `trace`, if given a list, records every goal visited in
    order."""
    if visited is None:
        visited = set()
    if trace is not None:
        trace.append(str(goal))
    if goal in visited:
        return False  # this goal is already being proved further up the chain: cycle
    visited = visited | {goal}

    facts = [c for c in kb.clauses if is_prop_symbol(c.op)]
    if goal in facts:
        return True

    for rule in kb.clauses:
        if rule.op == '==>' and rule.args[1] == goal:
            premises = conjuncts(rule.args[0])
            if all(pl_bc_ask(kb, p, visited, trace) for p in premises):
                return True
    return False

In [10]:
# Same rule set as 4.3, dibuktikan mundur dari Q.
trace = []
result = pl_bc_ask(horn_clauses_KB, expr('Q'), trace=trace)
print("Q entailed?", result)
print("Goal yang dikunjungi, berurutan:", trace)

Q entailed? True
Goal yang dikunjungi, berurutan: ['Q', 'P', 'L', 'A', 'P', 'A', 'B', 'M', 'B', 'L', 'A', 'P', 'A', 'B']


In [11]:
# Rule set siklik dari slide: A ==> B, B ==> C, C ==> A. Buktikan A.
cycle_kb = PropDefiniteKB()
for clause in ['A ==> B', 'B ==> C', 'C ==> A']:
    cycle_kb.tell(expr(clause))

trace = []
result = pl_bc_ask(cycle_kb, expr('A'), trace=trace)
print("A entailed?", result)
print("Goal yang dikunjungi:", trace)

A entailed? False
Goal yang dikunjungi: ['A', 'C', 'B', 'A']


Hasilnya `False`: tidak ada fact tunggal di `cycle_kb` sama sekali, jadi
`A` memang tidak seharusnya terbukti. Yang penting bukan hasil akhirnya,
tapi bahwa rekursinya **berhenti** dengan rapi lewat `visited`, bukan
infinite loop -- perhatikan `A` muncul lagi di `trace` saat mencoba
membuktikan premise `C ==> A`, dan begitu `goal in visited` terpenuhi,
cabang itu langsung dipotong.

---
# Latihan Soal

## Soal 1

Backward chaining (4.4) butuh `visited`/penjagaan loop, sedangkan forward
chaining (4.3) tidak. Jelaskan kenapa, dengan merujuk ke bagaimana
masing-masing algoritma bergerak (arah data-driven vs goal-driven,
rekursif atau tidak).

<details>
<summary>Klik untuk melihat hint</summary>

Lihat lagi bagaimana `pl_fc_entails`/`pl_fc_entails_traced` bergerak
(4.3): apakah dia pernah memproses satu fact lebih dari sekali, atau
"memanggil balik" fact lama? Bandingkan dengan `pl_bc_ask` (4.4), yang
memanggil dirinya sendiri secara rekursif untuk tiap premise.

Coba telusuri manual `trace` dari contoh `cycle_kb` di 4.4: goal apa yang
muncul **dua kali** di list itu? Kalau `visited` tidak ada, apa yang
akan terjadi persis di titik itu?

</details>

## Soal 2

`logic.py` juga menyediakan `definite_clauses_KB`, rule set lain (fact
`A`, `B`, `C`, lalu rule sampai ke `J`). Jalankan **forward chaining**
(`pl_fc_entails_traced`) untuk membuktikan `H`, tampilkan trace-nya, dan
tulis satu kalimat yang menyimpulkan fact apa saja yang berhasil
diturunkan sebelum `H` ditemukan.

<details>
<summary>Klik untuk melihat hint</summary>

Lihat dulu isi `definite_clauses_KB.clauses` satu per satu (`for clause
in ...: print(clause)`). Dari fact `A`, `B`, `C`, rule mana saja yang
**semua** premise-nya langsung terpenuhi? Telusuri manual satu-dua
langkah dulu di atas kertas sebelum menjalankan `pl_fc_entails_traced` --
lalu cocokkan urutan di `trace` dengan penelusuranmu.

Ada dua fact lain yang bisa diturunkan (`D` dan satu lagi) yang sama
sekali tidak diperlukan untuk sampai ke `H` -- perhatikan apakah mereka
tetap muncul di trace juga, dan pikirkan kenapa.

</details>

## Soal 3

Masih dengan `definite_clauses_KB`, buktikan **`J`** kali ini, pakai
`pl_bc_ask` (4.4). Jelaskan lewat trace-nya kenapa `J` gagal dibuktikan --
rule mana yang premise-nya tidak pernah terpenuhi?

<details>
<summary>Klik untuk melihat hint</summary>

Cari rule yang konklusinya `J` -- cuma ada satu. Rule itu punya berapa
premise? Coba buktikan tiap premise-nya satu per satu secara manual: ada
di daftar fact langsung, atau perlu rule lain lagi? Salah satu premise
itu tidak akan pernah bisa dibuktikan dari `definite_clauses_KB` -- cari
yang mana, baru jalankan `pl_bc_ask` dengan `trace=[]` untuk konfirmasi.

</details>

## Soal 4

Bandingkan forward chaining dan backward chaining pada
`definite_clauses_KB` untuk query `H` (Soal 2). Hitung berapa banyak fact
yang ditambahkan forward chaining ke `agenda` sebelum `H` ditemukan
(lihat jumlah baris di `trace` Soal 2), lalu bandingkan dengan berapa
banyak goal yang dikunjungi `pl_bc_ask` untuk membuktikan `H` secara
langsung. Mana yang memeriksa lebih sedikit clause untuk KB dan query
ini? Lalu, secara umum, pada kondisi apa forward chaining lebih cocok
dipakai, dan pada kondisi apa backward chaining lebih cocok? Jawaban
tidak cukup satu baris.

<details>
<summary>Klik untuk melihat hint</summary>

Hitung dari trace Soal 2: berapa fact TOTAL yang diproses forward
chaining sebelum berhenti (termasuk yang ternyata tidak dibutuhkan buat
`H`)? Lalu jalankan `pl_bc_ask(definite_clauses_KB, expr('H'),
trace=[])` dan hitung berapa goal yang muncul di `trace`-nya -- ada
selisih tidak, dan simbol apa saja yang disentuh satu tapi tidak
disentuh yang lain?

Perhatikan juga apakah ada satu goal yang muncul **lebih dari sekali**
di trace backward chaining -- kenapa itu bisa terjadi, padahal forward
chaining (`inferred[p]`) menjamin tiap fact cuma diproses sekali?

Untuk pertanyaan "kondisi apa cocok pakai yang mana": pikirkan skenario
KB yang sama tapi harus dijawab **banyak query berbeda** secara
berurutan -- apa hasil yang sudah dihitung forward chaining bisa dipakai
ulang? Bagaimana dengan backward chaining kalau query-nya cuma satu dan
sangat spesifik?

</details>